# IHC NDPI Quick Look

Explore the uploaded `.ndpi` immunohistochemistry slides without loading full-resolution whole-slide images into memory. The notebook reads embedded low-resolution NDPI overview images, creates contact sheets, saves individual quick-look PNGs, and writes slide metadata for planning dearraying/cropping.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

CURRENT_DIRECTORY = Path.cwd().resolve()
PIPELINE_ROOT = (
    CURRENT_DIRECTORY.parent
    if (CURRENT_DIRECTORY.parent / "src").exists()
    else CURRENT_DIRECTORY
)
PROJECT_ROOT = PIPELINE_ROOT.parent

IHC_DIRECTORY = PROJECT_ROOT / "immunohistochemistry"
OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / "immunohistochemistry_quicklook"
THUMBNAIL_DIRECTORY = OUTPUT_DIRECTORY / "thumbnails"

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
THUMBNAIL_DIRECTORY.mkdir(parents=True, exist_ok=True)

if not IHC_DIRECTORY.exists():
    raise FileNotFoundError(f"Could not find IHC upload directory: {IHC_DIRECTORY}")

ndpi_paths = sorted(IHC_DIRECTORY.glob("*.ndpi"))
if not ndpi_paths:
    raise FileNotFoundError(f"No .ndpi files found under {IHC_DIRECTORY}")

len(ndpi_paths), IHC_DIRECTORY, OUTPUT_DIRECTORY

## Helpers

The NDPI files have embedded series. In these uploads, `series[0]` is the full-resolution WSI, `series[1]` is a useful low-resolution macro overview, and `series[2]` is a small label image. The notebook uses the macro overview by default.

In [ ]:
def normalize_stain_name(value: str) -> str:
    """Normalize stain names for sorting and filenames."""

    cleaned = value.strip().replace("'", "")
    cleaned = cleaned.replace("NKX3.1", "NKX3_1")
    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", cleaned)
    return cleaned.strip("_").upper()


def parse_ndpi_name(path: Path) -> dict[str, str]:
    """Parse TMA block and stain from an NDPI filename."""

    sample_match = re.match(r"MET-TMA #(\w+)\s+(.+?)\s*\.ndpi$", path.name, flags=re.IGNORECASE)
    if sample_match:
        return {
            "slide_type": "sample",
            "tma_block": sample_match.group(1).upper(),
            "stain": normalize_stain_name(sample_match.group(2)),
            "display_name": f"TMA {sample_match.group(1).upper()} {normalize_stain_name(sample_match.group(2))}",
        }

    control_match = re.match(r"Control\s+(.+?)\s*\.ndpi$", path.name, flags=re.IGNORECASE)
    if control_match:
        stain = normalize_stain_name(control_match.group(1))
        return {
            "slide_type": "control",
            "tma_block": "CONTROL",
            "stain": stain,
            "display_name": f"Control {stain}",
        }

    return {
        "slide_type": "unknown",
        "tma_block": "UNKNOWN",
        "stain": normalize_stain_name(path.stem),
        "display_name": path.stem,
    }


def series_metadata(path: Path) -> list[dict[str, object]]:
    """Read lightweight series metadata from an NDPI file."""

    with tifffile.TiffFile(path) as tif:
        return [
            {
                "series_index": index,
                "shape": tuple(series.shape),
                "dtype": str(series.dtype),
                "axes": series.axes,
            }
            for index, series in enumerate(tif.series)
        ]


def read_overview_image(path: Path) -> np.ndarray:
    """Read the embedded low-resolution macro overview image."""

    with tifffile.TiffFile(path) as tif:
        if len(tif.series) > 1:
            image = tif.series[1].asarray()
        else:
            image = tif.series[-1].asarray()
    return np.asarray(image)


def safe_stem(value: str) -> str:
    """Return a safe filename stem."""

    return re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_")


def show_image(axis: plt.Axes, image: np.ndarray, title: str) -> None:
    """Display a grayscale or RGB image with consistent styling."""

    if image.ndim == 2:
        axis.imshow(image, cmap="gray")
    else:
        axis.imshow(image)
    axis.set_title(title, fontsize=9)
    axis.set_axis_off()

## Slide Metadata

This table tells us whether the upload is flat, which stains are present per TMA block, and what NDPI series are available.

In [ ]:
records = []
for path in ndpi_paths:
    parsed = parse_ndpi_name(path)
    metadata = series_metadata(path)
    records.append(
        {
            **parsed,
            "path": str(path),
            "filename": path.name,
            "bytes": path.stat().st_size,
            "n_series": len(metadata),
            "series_summary": "; ".join(
                f"{item['series_index']}:{item['shape']}:{item['axes']}" for item in metadata
            ),
        }
    )

slide_metadata = pd.DataFrame.from_records(records).sort_values(
    ["slide_type", "tma_block", "stain", "filename"]
)
metadata_path = OUTPUT_DIRECTORY / "slide_metadata.csv"
slide_metadata.to_csv(metadata_path, index=False)
slide_metadata

## Save Individual Overview Images

Each PNG is a quick-look macro overview, not a full-resolution export.

In [ ]:
thumbnail_records = []
for row in slide_metadata.itertuples(index=False):
    image = read_overview_image(Path(row.path))
    output_path = THUMBNAIL_DIRECTORY / f"{safe_stem(row.display_name)}.png"
    plt.imsave(output_path, image, cmap="gray" if image.ndim == 2 else None)
    thumbnail_records.append(
        {
            "display_name": row.display_name,
            "tma_block": row.tma_block,
            "stain": row.stain,
            "thumbnail_path": str(output_path),
            "thumbnail_shape": tuple(image.shape),
        }
    )

thumbnail_metadata = pd.DataFrame.from_records(thumbnail_records)
thumbnail_metadata.to_csv(OUTPUT_DIRECTORY / "thumbnail_metadata.csv", index=False)
thumbnail_metadata.head()

## Contact Sheet by TMA Block and Stain

This is the fastest way to see whether each stain has the same spot layout, whether spots are missing, and whether the slides are whole TMA images rather than already core-cropped images.

In [ ]:
sample_metadata = slide_metadata[slide_metadata["slide_type"] == "sample"].copy()
tma_blocks = sorted(sample_metadata["tma_block"].unique())
stains = ["CD3", "CD56", "CD68", "MYE", "NKX3_1_PANCK", "PAX5"]

fig, axes = plt.subplots(
    len(tma_blocks),
    len(stains),
    figsize=(4.0 * len(stains), 2.0 * len(tma_blocks)),
    squeeze=False,
)
for row_index, block in enumerate(tma_blocks):
    for column_index, stain in enumerate(stains):
        axis = axes[row_index, column_index]
        matching = sample_metadata[
            (sample_metadata["tma_block"] == block) & (sample_metadata["stain"] == stain)
        ]
        if matching.empty:
            axis.set_title(f"TMA {block} {stain}\nmissing", fontsize=9)
            axis.set_axis_off()
            continue
        image = read_overview_image(Path(matching.iloc[0]["path"]))
        show_image(axis, image, f"TMA {block} {stain}")

fig.tight_layout()
contact_sheet_path = OUTPUT_DIRECTORY / "sample_tma_stain_contact_sheet.png"
fig.savefig(contact_sheet_path, dpi=180, bbox_inches="tight")
plt.show()
contact_sheet_path

## Control Slides

The controls are shown separately because they do not map to a TMA block or Xenium region.

In [ ]:
control_metadata = slide_metadata[slide_metadata["slide_type"] == "control"].copy()
if control_metadata.empty:
    print("No control slides found.")
else:
    n_columns = min(3, len(control_metadata))
    n_rows = math.ceil(len(control_metadata) / n_columns)
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(5 * n_columns, 2.6 * n_rows), squeeze=False)
    for axis in axes.ravel():
        axis.set_axis_off()
    for axis, row in zip(axes.ravel(), control_metadata.itertuples(index=False)):
        image = read_overview_image(Path(row.path))
        show_image(axis, image, row.display_name)
    fig.tight_layout()
    control_contact_sheet_path = OUTPUT_DIRECTORY / "control_contact_sheet.png"
    fig.savefig(control_contact_sheet_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(control_contact_sheet_path)

## A Closer Look at One TMA Block

Change `selected_tma_block` to inspect one block across all stains. This is useful for checking spot correspondence across stains before deciding how to crop cores.

In [ ]:
selected_tma_block = "2C"
selected = sample_metadata[sample_metadata["tma_block"] == selected_tma_block].copy()
if selected.empty:
    raise ValueError(f"No slides found for TMA block {selected_tma_block}")

selected = selected.set_index("stain").reindex(stains).dropna(subset=["path"]).reset_index()
fig, axes = plt.subplots(2, 3, figsize=(18, 7), squeeze=False)
for axis in axes.ravel():
    axis.set_axis_off()
for axis, row in zip(axes.ravel(), selected.itertuples(index=False)):
    image = read_overview_image(Path(row.path))
    show_image(axis, image, f"TMA {selected_tma_block} {row.stain}")
fig.tight_layout()
selected_block_path = OUTPUT_DIRECTORY / f"TMA_{selected_tma_block}_all_stains_quicklook.png"
fig.savefig(selected_block_path, dpi=220, bbox_inches="tight")
plt.show()
selected_block_path

## Optional Manual Thumbnail Crop

Use this cell to quickly crop a region of the low-resolution overview by pixel coordinates. This does not crop the full-resolution WSI; it is only for visually testing approximate spot locations.

In [ ]:
crop_tma_block = "2C"
crop_stain = "NKX3_1_PANCK"
crop_bounds = None  # Example: (850, 180, 1800, 580) as (x0, y0, x1, y1)

crop_row = sample_metadata[
    (sample_metadata["tma_block"] == crop_tma_block)
    & (sample_metadata["stain"] == crop_stain)
]
if crop_row.empty:
    raise ValueError(f"No slide found for TMA {crop_tma_block} / {crop_stain}")

image = read_overview_image(Path(crop_row.iloc[0]["path"]))
if crop_bounds is None:
    cropped = image
    crop_title = "Full overview; set crop_bounds to inspect a subregion"
else:
    x0, y0, x1, y1 = crop_bounds
    cropped = image[y0:y1, x0:x1]
    crop_title = f"Crop x={x0}:{x1}, y={y0}:{y1}"

fig, axis = plt.subplots(figsize=(12, 4))
show_image(axis, cropped, f"TMA {crop_tma_block} {crop_stain}\n{crop_title}")
fig.tight_layout()
plt.show()

## Output Summary

Generated files are stored under `output/immunohistochemistry_quicklook/`.

In [ ]:
summary = {
    "metadata_csv": metadata_path,
    "thumbnail_metadata_csv": OUTPUT_DIRECTORY / "thumbnail_metadata.csv",
    "sample_contact_sheet": OUTPUT_DIRECTORY / "sample_tma_stain_contact_sheet.png",
    "control_contact_sheet": OUTPUT_DIRECTORY / "control_contact_sheet.png",
    "thumbnail_directory": THUMBNAIL_DIRECTORY,
}
for label, path in summary.items():
    print(f"{label}: {path}")